In [ ]:
import numpy as np
from functools import wraps
from typing import overload
from pathlib import Path
import datetime as dt
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
import random
from notebook_utils import get_parent_dir
random.seed(42)
np.random.seed(42)

In [ ]:
IN_DIR = get_parent_dir() / "In"
OUT_DIR = get_parent_dir() / "Out"

In [ ]:
results_df=pd.read_csv(OUT_DIR / "predictions_model_abstract_dev.csv", dtype={"label": str, "patient_id": str})
results_df["date"] = pd.to_datetime(results_df.date)

In [ ]:
results_df.dtypes

In [ ]:
results_df.head();

In [ ]:
pts = list(np.random.choice(results_df.patient_id.unique(), replace=False, size=10))

In [ ]:
sample_pt = results_df[results_df.patient_id.isin(pts)]

In [ ]:
sample_pt.groupby('patient_id')["label"].max()

In [ ]:
(
    so.Plot(data=sample_pt, x="date", y="proba", color="patient_id", group="label")
    .add(so.Line())
)

In [ ]:
def max_ll(pt_dates_df):
    
    pt_dates_df = pt_dates_df.sort_values("date")
    pt_dates_df = pt_dates_df.reset_index()
    likelihood_df = pt_dates_df.copy()
    likelihood_df["no_event_pred"] = np.log10(1 - likelihood_df["proba"])
    likelihood_df["event_pred"] = np.log10(likelihood_df["proba"])
    likelihood = [None] * likelihood_df.shape[0]
    likelihood_no_event = sum(likelihood_df["no_event_pred"])
    
    for i in likelihood_df.index:
        likelihood[i] = sum(likelihood_df["no_event_pred"][:i]) + sum(likelihood_df["event_pred"][i:])
    
    likelihood_df["likelihood"] = likelihood
    max_likelihood = max(likelihood_df["likelihood"])
    max_index = np.where(likelihood_df["likelihood"] == max_likelihood)[0][0]
    
    if max_likelihood > likelihood_no_event:
        optimal_date = likelihood_df["date"][max_index]
    else:
        optimal_date = None
    # print(likelihood_df)
    return(optimal_date)


In [ ]:
vte_dates = results_df.groupby('patient_id')[["text", "date", "proba"]].apply(max_ll).reset_index()
vte_dates.columns = ["patient_id", "predicted_date"]

In [ ]:
vte_dates.head()

In [ ]:
events = pd.read_parquet(IN_DIR / "prepped_core_7_21_2022.parquet",
                         columns=["MRN", "CANCER_VTE_DATE", "OLD_VTE_DATE"]
                         )
events["CANCER_VTE_DATE"] = pd.to_datetime(events["CANCER_VTE_DATE"])
events["patient_id"] = events.MRN.astype(str)

In [ ]:
events.head()

In [ ]:
events.dtypes

In [ ]:
predicted_dates_df = events.merge(vte_dates)[["patient_id", "CANCER_VTE_DATE", "predicted_date"]]

In [ ]:
predicted_dates_df.head()

In [ ]:
predicted_dates_df["diff"] = (predicted_dates_df.predicted_date - predicted_dates_df.CANCER_VTE_DATE).dt.days

In [ ]:
predicted_dates_df["diff"].head()

In [ ]:
predicted_dates_df.head()

In [ ]:
(
    so.Plot(data=predicted_dates_df, x="diff")
    .add(so.Bar(), so.Hist())
)

In [ ]:
results_df[results_df.patient_id=="376009"];

In [ ]:
results_df.loc[1369].text;

In [ ]:
results_df.head();

In [ ]:
diff_df = results_df.merge(predicted_dates_df[['patient_id', 'predicted_date', 'CANCER_VTE_DATE']])

In [ ]:
diff_df.shape

In [ ]:
diff_df["days_diff"] = (diff_df.date - diff_df.predicted_date).dt.days
diff_df["days_diff_actual"] = (diff_df.date - diff_df.CANCER_VTE_DATE).dt.days

In [ ]:
pred_actual_diff = (diff_df.predicted_date -   diff_df.CANCER_VTE_DATE).dt.days

In [ ]:
pred_actual_diff.mean()

In [ ]:
pred_actual_diff.median()

In [ ]:
p = pred_actual_diff.dropna()

In [ ]:
np.percentile(p, 80)

In [ ]:
p.describe()

In [ ]:
diff_df["bins_predicted"] = pd.cut(diff_df.days_diff, bins=range(-100, 200, 10), labels = range(-100, 200, 10)[1:] )
diff_df["bins_actual"] = pd.cut(diff_df.days_diff_actual, bins=range(-100, 200, 10), labels = range(-100, 200, 10)[1:] )

In [ ]:
diff_df.shape

In [ ]:
diff_df[(diff_df.days_diff_actual<-100) | (diff_df.days_diff_actual>200)].patient_id.nunique()

In [ ]:
diff_df.patient_id.nunique()

In [ ]:
diff_df[(diff_df.days_diff.isna()) & (diff_df.days_diff_actual.notna())].patient_id.nunique()

In [ ]:
from seaborn import axes_style
f, ax = plt.subplots(figsize=(14, 8))
ax.set_xlabel("a", fontsize=20, fontweight='bold')
ax.set_ylabel("a", fontsize=20, fontweight='bold')

(
    so.Plot(diff_df, x="bins_actual", y="proba")
    .add(so.Lines(), so.Agg())
    .add(so.Range(), so.Est(errorbar=('ci', 95), n_boot=1000, seed=42))
    .label(
        x="Days from Event",
        y="Probability",
        title="Mean Probability of CAT in Notes",
    )
    .theme({**axes_style("whitegrid"),
        # "legend.loc": "best",
        "font.weight": "bold",
        "font.size": 40,
        "axes.titlesize" : 20,
        "axes.titleweight": "bold"
       })
    .on(ax)
    .save("average_notes_probability_actual.svg",
       dpi=300,
       format="svg",
       bbox_inches="tight",
     )
);

In [ ]:
(
    so.Plot(diff_df, x="bins_predicted", y="proba")
    .add(so.Lines(), so.Agg())
    .add(so.Range(), so.Est(errorbar=('ci', 95), n_boot=1000, seed=42))
    .label(
        x="Days from Event",
        y="probability",
        color=str.capitalize,
        title="Mean Probability of CAT in Notes",
    )
    .layout(size=(12, 8))
    .save("average_notes_probability_predicted.svg",
       dpi=300,
       format="svg",
       bbox_inches="tight",
     )
);